# Import packages
Dependency Specification: pandas==3.0.5, duckdb==1.5.6

In [ ]:
from pathlib import Path
import pandas as pd
import duckdb

# Load Data
Load data from pre-downloaded Oracle Elixer's Data from 2014 to 2025 to a local persistent database file for quick access on future sessions. This is for training and testing simple models.

*Uncomment the code below if you don't have a duckdb yet.*

In [ ]:
# # Connect to persistent database file on disk
# con = duckdb.connect("lol_predictors.duckdb")

# oe_data_path = Path("./OE datasets")

# for dataset in oe_data_path.iterdir():
#     if dataset.is_file() and dataset.suffix == ".csv":
#         var_name = f"df_{dataset.name[:4]}" # e.g., df_2014
        
#         # Read CSV via pandas
#         df = pd.read_csv(dataset, low_memory=False)
        
#         # Save directly into a permanent DuckDB table in your file!
#         con.execute(f"CREATE OR REPLACE TABLE {var_name} AS SELECT * FROM df")
#         print(f"Saved {var_name} into DuckDB file.")

# con.close()

In [ ]:
con = duckdb.connect("lol_predictors.duckdb")

# Data Overview

Select relevant data

In [ ]:
def data_selection(df, included_cols):
    # Choose the dataset
    df = con.execute(f"""
    SELECT {", ".join(included_cols)}
    FROM {df}
    WHERE position = 'team';
    """).df()

    return df

In [ ]:
cols_to_include = ['gameid', 'league', 'year', 'split', 'patch',
                   'teamname', 'gamelength', 'result', 'kills',
                   'deaths', 'assists', 'towers', 'dpm', 'damagetakenperminute',
                   'wardskilled', 'visionscore', 'totalgold', 'goldspent'
                   ]

In [ ]:
df_selected = "df_2014" # choose input next time, instead of a hardcoded df name
df = data_selection(df_selected, cols_to_include)

In [ ]:
def data_profiling(df, df_selected=df_selected):
    """
    Present the profile of the dataframe, including
    description, number of missing values, number of unique values, and unique values of object series
    """
    # Display the data info
    print(f"{f" INFORMATION OF {df_selected} ":-^50}")
    df.info()
    print(f"Data Shape: {df.shape}")
    display(df.describe())
    
    with pd.option_context('display.max_seq_items', None): # list all columns of the df with no limits
        print(f"List of Columns: {df.columns}")
    
    with pd.option_context('display.max_rows', None): # list all rows of the df with no limits
        # State the number of missing items for each column
        print(f"\n{" COUNT OF MISSING VALUES ":-^50}") 
        display(df.isnull().sum())

        print(f"\n{" COUNT OF UNIQUE VALUES ":-^50}")
        display(df.nunique())

        print(f"\n{" UNIQUE VALUES OF OBJECT SERIES ":-^50}")
        for object_column in df.select_dtypes(include=['object']):
            if object_column == 'gameid': # skip gameid series
                continue
            print("\n",df[object_column].value_counts())


In [ ]:
def detect_outliers():
    pass

In [ ]:
def visual_profile():
    pass

In [ ]:
con.close()